# Phase 2 — Large Model Baseline

**Goal:**
Establish baseline performance for financial sentiment classification using:

1. **Zero-shot `roberta-large-mnli`**
   - Used as a **before fine-tuning baseline**
   - Evaluated directly on our dataset without task-specific training

2. **Fine-tuned `roberta-large`**
   - Fine-tuned on our training split for 3-class financial sentiment classification
   - Used as the strong large-model baseline for comparison and routing

**Per-example outputs:**

For each example in the validation and test sets, we save:
- sentence
- true label
- predicted label
- predicted class probabilities
- confidence
- entropy
- margin
- correctness indicator


In [3]:
%%capture
!pip install -q transformers datasets accelerate scikit-learn seaborn matplotlib

In [4]:
!pip install -U pip
!pip install pandas numpy torch scikit-learn matplotlib seaborn tqdm transformers datasets accelerate
!pip install python-dotenv

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
    --------------------------------------- 0.0/1.8 MB 1.3 MB/s eta 0:00:02
   ----- ---------------------------------- 0.2/1.8 MB 2.9 MB/s eta 0:00:01
   ----------------- ---------------------- 0.8/1.8 MB 6.1 MB/s eta 0:00:01
   ------------------------------ --------- 1.4/1.8 MB 7.9 MB/s eta 0:00:01
   ---------------------------------------  1.8/1.8 MB 8.7 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 8.1 MB/s eta 0:00:00


ERROR: To modify pip, please run the following command:
D:\NLP Project\.venv\Scripts\python.exe -m pip install -U pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

d:\NLP_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

In [3]:
def detect_text_col(df):
    for c in ["sentence", "text", "content", "query"]:
        if c in df.columns:
            return c
    raise ValueError(f"No text column found. Columns: {list(df.columns)}")

def detect_label_col(df):
    for c in ["label", "labels", "target", "sentiment"]:
        if c in df.columns:
            return c
    raise ValueError(f"No label column found. Columns: {list(df.columns)}")

def detect_id_col(df):
    for c in ["id", "idx", "index"]:
        if c in df.columns:
            return c
    return None

def save_confusion_matrix(y_true_ids, y_pred_ids, labels, out_path, title):
    cm = confusion_matrix(y_true_ids, y_pred_ids, labels=list(range(len(labels))))
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest")
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=labels,
        yticklabels=labels,
        ylabel="True label",
        xlabel="Predicted label",
        title=title,
    )

    thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    fig.tight_layout()
    plt.savefig(out_path, bbox_inches="tight", dpi=200)
    plt.close(fig)

In [16]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

def run_zeroshot_large_baseline(
    experiment_name,
    test_path,
    output_subdir,
    model_name="roberta-large-mnli"
):
    candidate_labels = ["negative", "neutral", "positive"]

    # Repo root, not processed_data folder
    PROJECT_DIR = Path(r"D:\NLP_Project\Linguistically-aware-model-routing-for-financial-sentiment-analysis")
    output_dir = PROJECT_DIR / "Output" / "model_outputs" / "Baseline_Roberta_large" / output_subdir
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(test_path)

    text_col = detect_text_col(df)
    label_col = detect_label_col(df)
    id_col = detect_id_col(df)

    if id_col is None:
        df["id"] = np.arange(len(df))
        id_col = "id"

    label2id = {"negative": 0, "neutral": 1, "positive": 2}
    id2label = {0: "negative", 1: "neutral", 2: "positive"}

    # Normalize labels safely
    # if df[label_col].dtype == object:
    #     df[label_col] = df[label_col].astype(str).str.strip().str.lower()
    #     df["true_label"] = df[label_col]
    #     df["true_label_id"] = df[label_col].map(label2id)

    #     if df["true_label_id"].isna().any():
    #         bad_labels = df.loc[df["true_label_id"].isna(), label_col].unique()
    #         raise ValueError(f"Unexpected labels found: {bad_labels}")
        
    #     df["true_label_id"] = df["true_label_id"].astype(int)
    # else:
    #     df["true_label_id"] = df[label_col].astype(int)
    #     df["true_label"] = df["true_label_id"].map(id2label)
    label2id = {"negative": 0, "neutral": 1, "positive": 2}
    id2label = {0: "negative", 1: "neutral", 2: "positive"}

    raw_labels = df[label_col].astype(str).str.strip().str.lower()

    if set(raw_labels.unique()).issubset(set(label2id.keys())):
        df["true_label"] = raw_labels
        df["true_label_id"] = raw_labels.map(label2id).astype(int)
    else:
        df["true_label_id"] = pd.to_numeric(df[label_col], errors="raise").astype(int)
        df["true_label"] = df["true_label_id"].map(id2label)

        print(f"\n===== {experiment_name} =====")
        print("Shape:", df.shape)
        print("Text column:", text_col)
        print("Label column:", label_col)

    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=-1   # CPU
    )

    pred_labels = []
    confidences = []
    prob_negative = []
    prob_neutral = []
    prob_positive = []

    for text in df[text_col].tolist():
        result = classifier(text, candidate_labels)

        pred_label = result["labels"][0].strip().lower()
        pred_labels.append(pred_label)
        confidences.append(result["scores"][0])

        score_map = {label.strip().lower(): score for label, score in zip(result["labels"], result["scores"])}
        prob_negative.append(score_map["negative"])
        prob_neutral.append(score_map["neutral"])
        prob_positive.append(score_map["positive"])

    df["pred_label"] = pred_labels
    df["pred_label_id"] = df["pred_label"].map(label2id)

    if df["pred_label_id"].isna().any():
        bad_preds = df.loc[df["pred_label_id"].isna(), "pred_label"].unique()
        raise ValueError(f"Unexpected predicted labels found: {bad_preds}")

    df["pred_label_id"] = df["pred_label_id"].astype(int)
    df["confidence"] = confidences
    df["prob_negative"] = prob_negative
    df["prob_neutral"] = prob_neutral
    df["prob_positive"] = prob_positive
    df["correct"] = (df["true_label"] == df["pred_label"]).astype(int)

    y_true_ids = df["true_label_id"].values
    y_pred_ids = df["pred_label_id"].values

    acc = accuracy_score(y_true_ids, y_pred_ids)
    macro_f1 = f1_score(y_true_ids, y_pred_ids, average="macro")

    print("Accuracy:", acc)
    print("Macro-F1:", macro_f1)
    print("\nPer-class report:\n")
    print(classification_report(
        y_true_ids,
        y_pred_ids,
        target_names=["negative", "neutral", "positive"]
    ))

    save_df = df[[
        id_col,
        text_col,
        "true_label_id",
        "true_label",
        "pred_label_id",
        "pred_label",
        "confidence",
        "prob_negative",
        "prob_neutral",
        "prob_positive",
        "correct"
    ]].copy()

    save_df = save_df.rename(columns={text_col: "sentence"})
    save_df.to_csv(output_dir / "test_predictions.csv", index=False)

    with open(output_dir / "test_metrics.json", "w") as f:
        json.dump({
            "experiment_name": experiment_name,
            "model": model_name,
            "setting": "before_finetuning_zeroshot",
            "accuracy": float(acc),
            "macro_f1": float(macro_f1),
            "num_examples": int(len(save_df))
        }, f, indent=2)

    save_confusion_matrix(
        y_true_ids,
        y_pred_ids,
        labels=["negative", "neutral", "positive"],
        out_path=output_dir / "test_confusion_matrix.png",
        title=f"{experiment_name} - Zero-shot RoBERTa-large"
    )

    return {
        "experiment_name": experiment_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "output_dir": str(output_dir)
    }

In [17]:
phrasebank_baseline = run_zeroshot_large_baseline(
    experiment_name="Baseline 1: Zero-Shot RoBERTa-large - PhraseBank Test Set",
    test_path="D:\\NLP_Project\Linguistically-aware-model-routing-for-financial-sentiment-analysis\data\processed_data\\test.csv",
    output_subdir="with_PhraseBank_Dataset"
)

<>:3: SyntaxWarning: invalid escape sequence '\L'
<>:3: SyntaxWarning: invalid escape sequence '\L'
C:\Users\IRONMAN\AppData\Local\Temp\ipykernel_9032\1209393610.py:3: SyntaxWarning: invalid escape sequence '\L'
  test_path="D:\\NLP_Project\Linguistically-aware-model-routing-for-financial-sentiment-analysis\data\processed_data\\test.csv",
d:\NLP_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\IRONMAN\.cache\huggingface\hub\models--roberta-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mod

Accuracy: 0.5791505791505791
Macro-F1: 0.5824545018460913

Per-class report:

              precision    recall  f1-score   support

    negative       0.42      1.00      0.59        63
     neutral       0.89      0.42      0.57       322
    positive       0.47      0.77      0.59       133

    accuracy                           0.58       518
   macro avg       0.60      0.73      0.58       518
weighted avg       0.73      0.58      0.58       518



In [ ]:
combined_baseline = run_zeroshot_large_baseline(
    experiment_name="Baseline 2: Zero-Shot RoBERTa-large - Combined Test Set",
    test_path="D:\\NLP_Project\Linguistically-aware-model-routing-for-financial-sentiment-analysis\data/combined_data/test.csv",
    output_subdir="with_Combined_Dataset"
)

In [ ]:
summary_df = pd.DataFrame([
    {
        "model": "roberta-large-mnli",
        "setting": "before_finetuning",
        "dataset": "PhraseBank",
        "accuracy": phrasebank_baseline["accuracy"],
        "macro_f1": phrasebank_baseline["macro_f1"]
    },
    {
        "model": "roberta-large-mnli",
        "setting": "before_finetuning",
        "dataset": "Combined",
        "accuracy": combined_baseline["accuracy"],
        "macro_f1": combined_baseline["macro_f1"]
    }
])

summary_df